# Project 1 — MLP in Pure NumPy (no PyTorch)
```
input -> Linear -> ReLU -> Linear -> Softmax -> Cross-entropy
```
**Goal:** implement forward, loss, backprop, SGD + mini-batches by hand on digit classification.
**Answers:** *Where does the gradient actually come from?* (chain rule through softmax+CE).
Runs offline: tries torchvision MNIST, falls back to sklearn digits (8x8).

In [ ]:
import os  # need os to create results dir and check env
import random  # need random to seed python RNG for reproducibility
import time  # need time to measure training duration
import numpy as np  # need numpy as the only tensor engine for this project
import matplotlib.pyplot as plt  # need matplotlib to plot loss/accuracy curves
SEED = 42  # fix a seed constant so every run is reproducible
random.seed(SEED)  # seed python's RNG so shuffling is deterministic
np.random.seed(SEED)  # seed numpy's RNG so init + shuffling are deterministic
os.makedirs('./results', exist_ok=True)  # create results folder (no error if it exists)
print('numpy', np.__version__)  # print numpy version for the lab log

In [ ]:
def load_data():  # define loader so dataset logic stays in one place
    """Try MNIST (torchvision), else sklearn digits. Returns X_train, y_train, X_test, y_test, n_classes."""  # docstring explains contract
    try:  # try the real MNIST path first
        from torchvision import datasets  # import torchvision datasets lazily (optional dep)
        import numpy as np  # re-import numpy inside fallback-safe scope
        tr = datasets.MNIST(root='/tmp/data', train=True, download=True)  # download/train split of MNIST
        te = datasets.MNIST(root='/tmp/data', train=False, download=True)  # download/test split of MNIST
        Xtr = tr.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0  # flatten 28x28 -> 784 and scale to [0,1]
        ytr = tr.targets.numpy().astype(np.int64)  # convert train labels to int64 array
        Xte = te.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0  # same flatten+scale for test
        yte = te.targets.numpy().astype(np.int64)  # convert test labels to int64 array
        print(f'MNIST loaded: train {Xtr.shape}, test {Xte.shape}')  # log shapes to confirm load
        return Xtr, ytr, Xte, yte, 10  # return MNIST arrays + 10 classes
    except Exception as e:  # if anything fails (no internet), use fallback
        print(f'MNIST unavailable ({e}), using sklearn digits')  # log why we fell back
        from sklearn.datasets import load_digits  # import sklearn digits dataset
        from sklearn.model_selection import train_test_split  # import splitter for train/test
        d = load_digits()  # load 1797 8x8 digit images
        X = d.data.astype(np.float32) / 16.0  # scale pixel values (0..16) to [0,1]
        y = d.target.astype(np.int64)  # labels 0..9 as int64
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)  # 80/20 split, fixed seed
        print(f'digits loaded: train {Xtr.shape}, test {Xte.shape}')  # log fallback shapes
        return Xtr, ytr, Xte, yte, 10  # return fallback arrays + 10 classes
X_train, y_train, X_test, y_test, N_CLASSES = load_data()  # call loader once, unpack all five outputs
N_IN = X_train.shape[1]  # infer input dim from data (784 for MNIST, 64 for digits)
print('n_in =', N_IN, '| train =', X_train.shape, '| test =', X_test.shape)  # confirm dims before building model

In [ ]:
def relu(z):  # define ReLU activation used after hidden layer
    return np.maximum(0, z)  # elementwise max(0,z): passes positives, kills negatives
def relu_backward(dout, z):  # define ReLU backward: gradient wrt pre-activation
    dz = dout.copy()  # copy upstream grad so we don't mutate caller's array
    dz[z <= 0] = 0  # zero grad where neuron was off (derivative of ReLU is 0 for z<=0)
    return dz  # return gradient wrt z
def softmax(logits):  # define numerically-stable softmax over classes
    shifted = logits - logits.max(axis=1, keepdims=True)  # subtract row-max for stability (prevents exp overflow)
    exp = np.exp(shifted)  # exponentiate shifted logits
    return exp / exp.sum(axis=1, keepdims=True)  # normalize each row to sum to 1 -> probabilities
def init_params(n_in, n_hid, n_out):  # He/Xavier-style init for the 2-layer MLP
    W1 = np.random.randn(n_in, n_hid).astype(np.float64) * np.sqrt(2.0 / n_in)  # He init: good for ReLU layers
    b1 = np.zeros(n_hid, dtype=np.float64)  # zero-init hidden bias (safe: symmetry broken by W1)
    W2 = np.random.randn(n_hid, n_out).astype(np.float64) * np.sqrt(1.0 / n_hid)  # Xavier-ish init for output layer
    b2 = np.zeros(n_out, dtype=np.float64)  # zero-init output bias
    return {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}  # pack params in dict for easy passing
N_HID = 128  # set hidden width: 128 is plenty for digits, fast on CPU
params = init_params(N_IN, N_HID, N_CLASSES)  # actually allocate W1,b1,W2,b2
print('W1', params['W1'].shape, 'W2', params['W2'].shape)  # sanity-check shapes: (n_in,128) and (128,10)

In [ ]:
def forward(params, X):  # define forward pass returning probs + cache for backprop
    Z1 = X @ params['W1'] + params['b1']  # Linear1: affine transform (N,H)
    A1 = relu(Z1)  # ReLU1: nonlinearity, cached as Z1 needed for backward
    logits = A1 @ params['W2'] + params['b2']  # Linear2: map hidden -> class scores (N,C)
    probs = softmax(logits)  # Softmax: scores -> probabilities
    cache = (X, Z1, A1, logits, probs)  # stash everything backward needs (X,Z1,A1,probs)
    return probs, cache  # return predictions + cache
def loss_and_grad_logits(probs, y):  # combined softmax+CE backward (the famous simplification)
    N = probs.shape[0]  # get batch size for mean-averaging
    loss = -np.log(probs[np.arange(N), y] + 1e-12).mean()  # cross-entropy: -mean log prob of true class
    dlogits = probs.copy()  # start from probs (dL/dlogits = p - onehot, derived via chain rule)
    dlogits[np.arange(N), y] -= 1  # subtract 1 at true class: p_k - 1[y==k]
    dlogits /= N  # divide by N because loss was a mean, not a sum
    return loss, dlogits  # return scalar loss + grad wrt logits
probs, cache = forward(params, X_train[:8])  # smoke-test forward on 8 samples
loss, dlogits = loss_and_grad_logits(probs, y_train[:8])  # smoke-test loss+grad on same 8
print(f'smoke test loss={loss:.4f}, probs row sums={probs.sum(axis=1)[:3]}')  # loss should be ~ln(10)=2.3; rows sum to 1

In [ ]:
def backward(params, cache, y, dlogits):  # define full backprop: logits -> W2/b2 -> A1 -> Z1 -> W1/b1
    X, Z1, A1, logits, probs = cache  # unpack cached forward tensors
    dW2 = A1.T @ dlogits  # grad W2 = A1^T @ dlogits (linear-layer pullback)
    db2 = dlogits.sum(axis=0)  # grad b2 = sum over batch (bias broadcasts)
    dA1 = dlogits @ params['W2'].T  # pull grad back through Linear2 into hidden activations
    dZ1 = relu_backward(dA1, Z1)  # apply ReLU mask: kill grad where Z1<=0
    dW1 = X.T @ dZ1  # grad W1 = X^T @ dZ1
    db1 = dZ1.sum(axis=0)  # grad b1 = sum over batch
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}  # pack grads mirroring params dict
def accuracy(params, X, y):  # helper: fraction correct
    probs, _ = forward(params, X)  # run forward (ignore cache)
    pred = probs.argmax(axis=1)  # predicted class = argmax probability
    return (pred == y).mean()  # mean of correct indicators
grads = backward(params, cache, y_train[:8], dlogits)  # smoke-test backward on the 8-sample batch
print('grad shapes:', {k: v.shape for k, v in grads.items()})  # each grad must match its param shape
print(f'init acc train={accuracy(params, X_train[:500], y_train[:500]):.3f}')  # random net ~= 10% (chance)

In [ ]:
LR = 0.1  # set learning rate for vanilla SGD (tune: 0.01..0.5)
EPOCHS = 15  # set number of full passes over training data
BATCH = 64  # set mini-batch size: 32..256 typical; 64 is a good default
n = X_train.shape[0]  # number of training samples
history = {'loss': [], 'acc': []}  # create log to record per-epoch loss/acc
t0 = time.time()  # start wall-clock timer for training
for epoch in range(1, EPOCHS + 1):  # loop over epochs (1-indexed for pretty printing)
    perm = np.random.permutation(n)  # shuffle indices each epoch so batches are iid-ish
    X_train, y_train = X_train[perm], y_train[perm]  # apply permutation in-place order (reassign views)
    ep_loss, nb = 0.0, 0  # reset epoch loss accumulator + batch counter
    for i in range(0, n, BATCH):  # walk through data in BATCH-sized chunks
        xb = X_train[i:i+BATCH]  # slice input mini-batch
        yb = y_train[i:i+BATCH]  # slice matching labels
        probs, cache = forward(params, xb)  # FORWARD: compute predictions
        loss, dlogits = loss_and_grad_logits(probs, yb)  # LOSS+SEED-GRAD: scalar loss + dL/dlogits
        grads = backward(params, cache, yb, dlogits)  # BACKWARD: chain rule to every param
        for k in params:  # loop over each parameter array
            params[k] -= LR * grads[k]  # SGD STEP: w -= lr * grad (in place)
        ep_loss += loss * len(xb)  # weight batch loss by batch size for correct mean
        nb += len(xb)  # count samples seen (last batch may be smaller)
    ep_loss /= nb  # divide to get mean epoch loss
    acc = accuracy(params, X_test, y_test)  # evaluate on held-out test set
    history['loss'].append(ep_loss)  # log epoch loss
    history['acc'].append(acc)  # log epoch test accuracy
    print(f'epoch {epoch:02d}/{EPOCHS} loss={ep_loss:.4f} test_acc={acc:.4f}')  # print progress line
dt = time.time() - t0  # stop timer
print(f'trained in {dt:.1f}s ({n*EPOCHS/dt:.0f} samples/sec)')  # report throughput: total samples / seconds

In [ ]:
import csv  # need csv to write results table portably
train_acc = accuracy(params, X_train, y_train)  # final train accuracy (detects over/underfit gap)
test_acc = accuracy(params, X_test, y_test)  # final test accuracy (the headline metric)
print(f'FINAL train_acc={train_acc:.4f} test_acc={test_acc:.4f}')  # print both for the gap comparison
with open('./results/metrics.csv', 'w', newline='') as f:  # open CSV for writing results
    w = csv.writer(f)  # create csv writer object
    w.writerow(['epoch', 'loss', 'test_acc'])  # write header row
    for e, (l, a) in enumerate(zip(history['loss'], history['acc']), 1):  # loop epochs 1-indexed
        w.writerow([e, f'{l:.6f}', f'{a:.6f}'])  # write one row per epoch
print('saved ./results/metrics.csv')  # confirm CSV path
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))  # create side-by-side axes for loss and acc
ax[0].plot(history['loss'], marker='o')  # plot loss vs epoch with markers
ax[0].set(title='Train loss (mean CE)', xlabel='epoch', ylabel='loss')  # label loss subplot
ax[1].plot(history['acc'], marker='o')  # plot test accuracy vs epoch
ax[1].set(title='Test accuracy', xlabel='epoch', ylabel='acc')  # label accuracy subplot
fig.tight_layout()  # avoid overlapping labels
fig.savefig('./results/curves.png', dpi=120)  # save figure for the README/results folder
plt.show()  # display figure inline in notebook

## What to write in your README
1. What did I build? 2. How does forward/backward work? 3. What accuracy/time? 4. What surprised you?
5. Bottleneck? (hint: Python-loop matmuls vs BLAS, batch size). 6. What if LR x10 / hidden 32? 7. Next: momentum/Adam?
**Gradient origin (one-liner):** `dL/dlogits = (softmax - onehot)/N` — softmax Jacobian times CE gradient collapses to this; everything else is linear pullbacks + ReLU masking.